In [11]:
import pandas as pd

archivo_gigante = "dataset.csv"   
archivo_pequeno = "dataset_val.csv"


In [12]:
print("Cargando el dataset maestro...")
df = pd.read_csv(archivo_gigante, encoding="utf-8-sig")
df.columns = df.columns.str.strip()

print(f"Filas cargadas: {len(df)}")
print(f"Columnas cargadas: {df.shape[1]}")

if len(df) == 0:
    raise ValueError("El CSV se cargó vacío. Revisa el archivo, el separador o el encoding.")


Cargando el dataset maestro...


C:\Users\samue\AppData\Local\Temp\ipykernel_21624\41560741.py:2: DtypeWarning: Columns (3,6,7,15,17,18,19,20,25,27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_gigante, encoding="utf-8-sig")


Filas cargadas: 144832
Columnas cargadas: 30


In [15]:

objetivo_total = 2750

df["_idx_original"] = df.index

columnas_estratificacion = ["fuente", "somnolencia", "gafas", "genero"]

grupo = df.groupby(columnas_estratificacion, dropna=False, sort=False)
n_grupos = grupo.ngroups

print(f"Número de subgrupos encontrados: {n_grupos}")

if n_grupos == 0:
    print("No se encontraron subgrupos. Tomando muestra aleatoria global.")
    df_reducido = df.sample(
        n=min(objetivo_total, len(df)),
        random_state=42
    )
else:

    base_por_grupo = max(1, objetivo_total // n_grupos)

    print(f"Extrayendo aproximadamente {base_por_grupo} registros por subgrupo...")

    muestras_seleccionadas = []

    for llaves_grupo, sub_dataset in grupo:
        cantidad_a_extraer = min(len(sub_dataset), base_por_grupo)

        if cantidad_a_extraer == 0:
            continue

        muestra_grupo = sub_dataset.sample(
            n=cantidad_a_extraer,
            random_state=42
        )

        muestras_seleccionadas.append(muestra_grupo)

        print(
            f"-> Extraídos {cantidad_a_extraer} registros de la combinación: "
            f"{dict(zip(columnas_estratificacion, llaves_grupo))}"
        )

    if len(muestras_seleccionadas) == 0:
        df_reducido = df.sample(
            n=min(objetivo_total, len(df)),
            random_state=42
        )
    else:
        df_reducido = pd.concat(muestras_seleccionadas, axis=0)

if len(df_reducido) < objetivo_total:
    faltantes = objetivo_total - len(df_reducido)

    print(f"\nFaltan {faltantes} filas para llegar a {objetivo_total}. Completando con muestras aleatorias...")

    usados = set(df_reducido["_idx_original"])
    disponibles = df[~df["_idx_original"].isin(usados)]

    if len(disponibles) > 0:
        extra = disponibles.sample(
            n=min(faltantes, len(disponibles)),
            random_state=42
        )
        df_reducido = pd.concat([df_reducido, extra], axis=0)

if len(df_reducido) > objetivo_total:
    df_reducido = df_reducido.sample(
        n=objetivo_total,
        random_state=42
    )

df_reducido = df_reducido.sample(frac=1, random_state=42).reset_index(drop=True)

df_reducido = df_reducido.drop(columns=["_idx_original"], errors="ignore")

columnas_finales = [
    "id", "fuente", "sujeto", "ruta_ojo", "ojo", "gafas", "genero",
    "iluminacion", "reflejo", "sensor", "hash", "atipico", "particion",
    "particion_aleatoria", "balanceado", "video", "fotograma",
    "ruta_ojo_der", "ruta_ojo_izq", "ruta_boca", "ruta_rostro",
    "ear_der", "ear_izq", "mar", "bostezo", "situacion", "pliegue",
    "ventana", "kss", "somnolencia"
]

columnas_existentes = [c for c in columnas_finales if c in df_reducido.columns]
columnas_faltantes = [c for c in columnas_finales if c not in df_reducido.columns]

if columnas_faltantes:
    print("\nEstas columnas finales no existen en el CSV y se omitirán:")
    print(columnas_faltantes)

df_reducido = df_reducido[columnas_existentes]

archivo_reducido = "dataset_pruebas_orquestador.csv"

df_reducido.to_csv(archivo_reducido, index=False)

print(f"\n¡Listo! Se creó '{archivo_reducido}' con {len(df_reducido)} filas.")

Número de subgrupos encontrados: 13
Extrayendo aproximadamente 211 registros por subgrupo...
-> Extraídos 211 registros de la combinación: {'fuente': 'mrl', 'somnolencia': nan, 'gafas': 0.0, 'genero': 'hombre'}
-> Extraídos 211 registros de la combinación: {'fuente': 'mrl', 'somnolencia': nan, 'gafas': 1.0, 'genero': 'hombre'}
-> Extraídos 211 registros de la combinación: {'fuente': 'mrl', 'somnolencia': nan, 'gafas': 0.0, 'genero': 'mujer'}
-> Extraídos 211 registros de la combinación: {'fuente': 'mrl', 'somnolencia': nan, 'gafas': 1.0, 'genero': 'mujer'}
-> Extraídos 211 registros de la combinación: {'fuente': 'yawdd', 'somnolencia': nan, 'gafas': 0.0, 'genero': 'mujer'}
-> Extraídos 211 registros de la combinación: {'fuente': 'yawdd', 'somnolencia': nan, 'gafas': 0.0, 'genero': 'hombre'}
-> Extraídos 211 registros de la combinación: {'fuente': 'yawdd', 'somnolencia': nan, 'gafas': 2.0, 'genero': 'hombre'}
-> Extraídos 211 registros de la combinación: {'fuente': 'yawdd', 'somnolencia